# 02 – Cleaning the laps (Bahrain 2022 and Monaco 2022)

In notebook 01 I explored the raw lap tables and wrote down what needs cleaning. Here I apply the functions of `f1_telemetry.cleaning` to the two snapshots and check that the results agree with what I found by hand. The logic lives in the package; this notebook only calls it and looks at the output.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

# The kernel may start in notebooks/ or in the project root: look upwards
# for pyproject.toml, which only exists in the root.
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)

# Let Python find the package, which lives in src/ (no pip install needed)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

## 1. Load the snapshots

I read the CSV snapshots saved in notebook 01, so this notebook needs no internet. The round numbers are written directly (Bahrain is round 1, Monaco round 7 in 2022). `load_dataframe` restores the time columns and keeps `DriverNumber` and `TrackStatus` as text.

In [3]:
import pandas as pd

from f1_telemetry.cleaning import (
    build_stints,
    get_clean_laps,
    is_wet_race,
    prepare_laps,
    wet_lap_share,
)
from f1_telemetry.data_loader import (
    LAP_DATETIME_COLUMNS,
    LAP_TIMEDELTA_COLUMNS,
    load_dataframe,
    snapshot_path,
)

YEAR = 2022
ROUNDS = {"Bahrain": 1, "Monaco": 7}

raw = {}
for name, round_number in ROUNDS.items():
    path = snapshot_path(YEAR, round_number, "R", "laps")
    raw[name] = load_dataframe(
        path,
        timedelta_columns=LAP_TIMEDELTA_COLUMNS,
        datetime_columns=LAP_DATETIME_COLUMNS,
    )
    print(name, raw[name].shape)

Bahrain (1125, 31)
Monaco (1179, 31)


## 2. Prepare the laps

`prepare_laps` returns a copy with integer `LapNumber` and `Stint`, one clean label for unknown compounds, a true/false `Deleted` and a new `LapTimeSeconds` column. It must not lose any row, so I compare the shapes. I also look at the compounds before and after, because Monaco has unknown tyres.

In [4]:
laps = {name: prepare_laps(df) for name, df in raw.items()}

for name in raw:
    print(name, "rows kept:", raw[name].shape[0] == laps[name].shape[0])

print("Monaco compounds, raw:")
print(raw["Monaco"]["Compound"].value_counts(dropna=False))
print("Monaco compounds, prepared:")
print(laps["Monaco"]["Compound"].value_counts())

laps["Bahrain"][["LapNumber", "Stint", "Compound", "Deleted", "LapTimeSeconds"]].dtypes

Bahrain rows kept: True
Monaco rows kept: True
Monaco compounds, raw:
Compound
HARD            393
MEDIUM          346
WET             283
INTERMEDIATE    115
NaN              35
SOFT              7
Name: count, dtype: int64
Monaco compounds, prepared:
Compound
HARD            393
MEDIUM          346
WET             283
INTERMEDIATE    115
UNKNOWN          35
SOFT              7
Name: count, dtype: int64


LapNumber           int64
Stint               int64
Compound           object
Deleted              bool
LapTimeSeconds    float64
dtype: object

## 3. Clean laps

`get_clean_laps` keeps the laps with a lap time that are not lap 1, not in or out laps, run under a clear track (`TrackStatus` exactly "1") and not deleted. In notebook 01 I found that every green lap that was not `IsAccurate` was either a lap 1 or a pit lap, so I expect all clean laps to be `IsAccurate`. I also count how many clean laps each driver has, because a driver with few clean laps cannot give a reliable degradation slope.

In [5]:
rows = []
for name, df in laps.items():
    clean = get_clean_laps(df)
    rows.append(
        {
            "Race": name,
            "Laps": len(df),
            "CleanLaps": len(clean),
            "ShareKept": round(len(clean) / len(df), 3),
            "AllAccurate": bool(clean["IsAccurate"].all()),
        }
    )
pd.DataFrame(rows)

,Race,Laps,CleanLaps,ShareKept,AllAccurate
0,Bahrain,1125,854,0.759,True
1,Monaco,1179,833,0.707,True


In [ ]:
clean_bahrain = get_clean_laps(laps["Bahrain"])
clean_bahrain.groupby("Driver").size().sort_values().head(8)

Driver
GAS    39
ALO    41
VER    41
NOR    41
TSU    42
HAM    42
HUL    42
STR    42
dtype: int64

## 4. Stints

`build_stints` gives one row per driver and stint, built from all laps (a stint includes its in and out laps). Two checks: the laps counted in the stints must add up to the lap rows of each driver, and a stint must have one single compound.

In [9]:
stints = {name: build_stints(df) for name, df in laps.items()}

for name in laps:
    in_stints = stints[name].groupby("Driver")["Laps"].sum()
    in_laps = laps[name].groupby("Driver").size()
    one_compound = laps[name].groupby(["Driver", "Stint"])["Compound"].nunique().max()
    print(
        name,
        "| stint laps match lap rows:",
        bool((in_stints == in_laps).all()),
        "| max compounds in one stint:",
        one_compound,
    )

stints["Bahrain"].head(8)

Bahrain | stint laps match lap rows: True | max compounds in one stint: 1
Monaco | stint laps match lap rows: True | max compounds in one stint: 1


,Driver,Stint,Compound,FirstLap,LastLap,Laps,TyreLifeStart,TyreLifeEnd,FreshTyre
0,ALB,1,SOFT,1,13,13,1.0,13.0,True
1,ALB,2,MEDIUM,14,35,22,1.0,22.0,True
2,ALB,3,MEDIUM,36,46,11,3.0,13.0,False
3,ALB,4,SOFT,47,57,11,4.0,14.0,False
4,ALO,1,SOFT,1,11,11,4.0,14.0,False
5,ALO,2,MEDIUM,12,25,14,1.0,14.0,True
6,ALO,3,HARD,26,42,17,1.0,17.0,True
7,ALO,4,SOFT,43,57,15,1.0,15.0,True


## 5. Wet or dry

A race is wet when at least 10% of its laps are on WET or INTERMEDIATE tyres. I expect Bahrain to be dry and Monaco, which I used as a stress test, to be wet.

In [10]:
for name, df in laps.items():
    print(
        name, "wet share:", round(wet_lap_share(df), 3), "| wet race:", is_wet_race(df)
    )

Bahrain wet share: 0.0 | wet race: False
Monaco wet share: 0.338 | wet race: True
